In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, adjusted_rand_score

In [2]:
#load the training methylation + clinical data (the holdout set is intentionally never used here)
clinical_df = pd.read_csv('../data/processed/clinical_kirc_train.csv')
methylation_df = pd.read_parquet('../data/processed/methylation_kirc_train.parquet')

#align clinical data to methylation's row order (clinical CSV order != methylation parquet order)
clinical_df = clinical_df.set_index('sample').reindex(methylation_df.index)

le = LabelEncoder()
y_true_all = le.fit_transform(clinical_df['tissue_type.samples'])
patient_ids = methylation_df.index.str[:12]

print(f"Total samples: {len(methylation_df)}, unique patients: {patient_ids.nunique()}")
print(f"Class balance: {dict(zip(le.classes_, np.bincount(y_true_all)))}")

Total samples: 285, unique patients: 191
Class balance: {'Normal': np.int64(92), 'Tumor': np.int64(193)}


In [ ]:
N_FOLDS = 5
# compare CV robustness across the same PCA dimensionalities 
N_COMPONENTS_LIST = [2, 3, 112]

gkf = GroupKFold(n_splits=N_FOLDS)

fold_results = []
X_all = methylation_df.values

for n_components in N_COMPONENTS_LIST:
    print(f"\n{'='*20} GMM, N_COMPONENTS = {n_components} {'='*20}")
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_all, y_true_all, groups=patient_ids)):
        print(f"\n=== Fold {fold_idx + 1}/{N_FOLDS} (n_components={n_components}) ===")

        X_fold_train, X_fold_val = X_all[train_idx], X_all[val_idx]
        y_fold_train, y_fold_val = y_true_all[train_idx], y_true_all[val_idx]

        #refit scaler + PCA + clustering WITHIN this fold only
        scaler = StandardScaler()
        X_fold_train_scaled = scaler.fit_transform(X_fold_train)
        X_fold_val_scaled = scaler.transform(X_fold_val)

        pca = PCA(n_components=n_components, random_state=42)
        X_fold_train_pca = pca.fit_transform(X_fold_train_scaled)
        X_fold_val_pca = pca.transform(X_fold_val_scaled)

        gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42, n_init=20)
        pseudo_labels_train = gmm.fit_predict(X_fold_train_pca)

        #cluster IDs are arbitrary -- align pseudo-label direction to ground truth via majority vote
        #(same fix applied in 04_classification.ipynb)
        if (pseudo_labels_train == y_fold_train).mean() < 0.5:
            pseudo_labels_train = 1 - pseudo_labels_train

        fold_ari = adjusted_rand_score(y_fold_train, pseudo_labels_train)

        #train classifiers on pseudo-labels and on ground truth, within this fold's training portion
        svm_pseudo = SVC(kernel='rbf', probability=True, random_state=67).fit(X_fold_train_pca, pseudo_labels_train)
        svm_truth = SVC(kernel='rbf', probability=True, random_state=67).fit(X_fold_train_pca, y_fold_train)
        rf_pseudo = RandomForestClassifier(n_estimators=100, random_state=67).fit(X_fold_train_pca, pseudo_labels_train)
        rf_truth = RandomForestClassifier(n_estimators=100, random_state=67).fit(X_fold_train_pca, y_fold_train)

        svm_auc_pseudo = roc_auc_score(y_fold_val, svm_pseudo.predict_proba(X_fold_val_pca)[:, 1])
        svm_auc_truth = roc_auc_score(y_fold_val, svm_truth.predict_proba(X_fold_val_pca)[:, 1])
        rf_auc_pseudo = roc_auc_score(y_fold_val, rf_pseudo.predict_proba(X_fold_val_pca)[:, 1])
        rf_auc_truth = roc_auc_score(y_fold_val, rf_truth.predict_proba(X_fold_val_pca)[:, 1])

        print(f"ARI: {fold_ari:.4f} | SVM AUC (pseudo/truth): {svm_auc_pseudo:.4f}/{svm_auc_truth:.4f} | RF AUC (pseudo/truth): {rf_auc_pseudo:.4f}/{rf_auc_truth:.4f}")

        fold_results.append({
            'n_components': f'gmm_{n_components}',
            'fold': fold_idx + 1,
            'ari': fold_ari,
            'svm_auc_pseudo': svm_auc_pseudo,
            'svm_auc_truth': svm_auc_truth,
            'rf_auc_pseudo': rf_auc_pseudo,
            'rf_auc_truth': rf_auc_truth,
        })

results_df = pd.DataFrame(fold_results)
results_df

In [ ]:
#created the original Raw-features-KMeans-label / PCA(3)-classifier-features
#construction with a simpler, more natural bad arm: UMAP(n=30) x Hierarchical, i.e. exactly the
#worst-performing REAL configuration already reported in 02-2_clustering_umap.ipynb's single-split
#table (ARI=0.018 there), using the SAME features for both the pseudo-label and the classifier (no
#mixed feature spaces to explain). 
BAD_LABEL_TAG = 'umap_hier_bad'

for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_all, y_true_all, groups=patient_ids)):
    print(f"\n=== Fold {fold_idx + 1}/{N_FOLDS} (UMAP(30) x Hierarchical, deliberately poor pseudo-label) ===")

    X_fold_train, X_fold_val = X_all[train_idx], X_all[val_idx]
    y_fold_train, y_fold_val = y_true_all[train_idx], y_true_all[val_idx]

    scaler = StandardScaler()
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler.transform(X_fold_val)

    #matches 02-2_clustering_umap.ipynb's optimal-n UMAP call exactly (n_components=30, random_state=42)
    reducer = umap.UMAP(n_components=30, random_state=42)
    X_fold_train_umap = reducer.fit_transform(X_fold_train_scaled)
    X_fold_val_umap = reducer.transform(X_fold_val_scaled)

    #matches 02-2_clustering_umap.ipynb's hierarchical call exactly (n_clusters=2, linkage='ward')
    hier = AgglomerativeClustering(n_clusters=2, linkage='ward')
    pseudo_labels_train = hier.fit_predict(X_fold_train_umap)
    if (pseudo_labels_train == y_fold_train).mean() < 0.5:
        pseudo_labels_train = 1 - pseudo_labels_train
    fold_ari = adjusted_rand_score(y_fold_train, pseudo_labels_train)

    #classifier trained on the SAME UMAP(30) features that were clustered -- a single natural
    #pipeline, unlike the old construction's separate label/feature spaces
    svm_pseudo = SVC(kernel='rbf', probability=True, random_state=67).fit(X_fold_train_umap, pseudo_labels_train)
    svm_truth = SVC(kernel='rbf', probability=True, random_state=67).fit(X_fold_train_umap, y_fold_train)
    rf_pseudo = RandomForestClassifier(n_estimators=100, random_state=67).fit(X_fold_train_umap, pseudo_labels_train)
    rf_truth = RandomForestClassifier(n_estimators=100, random_state=67).fit(X_fold_train_umap, y_fold_train)

    svm_auc_pseudo = roc_auc_score(y_fold_val, svm_pseudo.predict_proba(X_fold_val_umap)[:, 1])
    svm_auc_truth = roc_auc_score(y_fold_val, svm_truth.predict_proba(X_fold_val_umap)[:, 1])
    rf_auc_pseudo = roc_auc_score(y_fold_val, rf_pseudo.predict_proba(X_fold_val_umap)[:, 1])
    rf_auc_truth = roc_auc_score(y_fold_val, rf_truth.predict_proba(X_fold_val_umap)[:, 1])

    print(f"ARI: {fold_ari:.4f} | SVM AUC (pseudo/truth): {svm_auc_pseudo:.4f}/{svm_auc_truth:.4f} | RF AUC (pseudo/truth): {rf_auc_pseudo:.4f}/{rf_auc_truth:.4f}")

    fold_results.append({
        'n_components': BAD_LABEL_TAG,
        'fold': fold_idx + 1,
        'ari': fold_ari,
        'svm_auc_pseudo': svm_auc_pseudo,
        'svm_auc_truth': svm_auc_truth,
        'rf_auc_pseudo': rf_auc_pseudo,
        'rf_auc_truth': rf_auc_truth,
    })

results_df = pd.DataFrame(fold_results)
results_df

In [ ]:
#check if the three GMM arms above test whether the dimensionality effect and (later) the ARI-vs-AUC-gap relationship hold up under resampling
for n_components in N_COMPONENTS_LIST:
    print(f"\n{'='*20} KMeans, N_COMPONENTS = {n_components} {'='*20}")
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_all, y_true_all, groups=patient_ids)):
        print(f"\n=== Fold {fold_idx + 1}/{N_FOLDS} (KMeans, n_components={n_components}) ===")

        X_fold_train, X_fold_val = X_all[train_idx], X_all[val_idx]
        y_fold_train, y_fold_val = y_true_all[train_idx], y_true_all[val_idx]

        scaler = StandardScaler()
        X_fold_train_scaled = scaler.fit_transform(X_fold_train)
        X_fold_val_scaled = scaler.transform(X_fold_val)

        pca = PCA(n_components=n_components, random_state=42)
        X_fold_train_pca = pca.fit_transform(X_fold_train_scaled)
        X_fold_val_pca = pca.transform(X_fold_val_scaled)

        kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
        pseudo_labels_train = kmeans.fit_predict(X_fold_train_pca)
        if (pseudo_labels_train == y_fold_train).mean() < 0.5:
            pseudo_labels_train = 1 - pseudo_labels_train
        fold_ari = adjusted_rand_score(y_fold_train, pseudo_labels_train)

        svm_pseudo = SVC(kernel='rbf', probability=True, random_state=67).fit(X_fold_train_pca, pseudo_labels_train)
        svm_truth = SVC(kernel='rbf', probability=True, random_state=67).fit(X_fold_train_pca, y_fold_train)
        rf_pseudo = RandomForestClassifier(n_estimators=100, random_state=67).fit(X_fold_train_pca, pseudo_labels_train)
        rf_truth = RandomForestClassifier(n_estimators=100, random_state=67).fit(X_fold_train_pca, y_fold_train)

        svm_auc_pseudo = roc_auc_score(y_fold_val, svm_pseudo.predict_proba(X_fold_val_pca)[:, 1])
        svm_auc_truth = roc_auc_score(y_fold_val, svm_truth.predict_proba(X_fold_val_pca)[:, 1])
        rf_auc_pseudo = roc_auc_score(y_fold_val, rf_pseudo.predict_proba(X_fold_val_pca)[:, 1])
        rf_auc_truth = roc_auc_score(y_fold_val, rf_truth.predict_proba(X_fold_val_pca)[:, 1])

        print(f"ARI: {fold_ari:.4f} | SVM AUC (pseudo/truth): {svm_auc_pseudo:.4f}/{svm_auc_truth:.4f} | RF AUC (pseudo/truth): {rf_auc_pseudo:.4f}/{rf_auc_truth:.4f}")

        fold_results.append({
            'n_components': f'kmeans_{n_components}',
            'fold': fold_idx + 1,
            'ari': fold_ari,
            'svm_auc_pseudo': svm_auc_pseudo,
            'svm_auc_truth': svm_auc_truth,
            'rf_auc_pseudo': rf_auc_pseudo,
            'rf_auc_truth': rf_auc_truth,
        })

results_df = pd.DataFrame(fold_results)
results_df

In [5]:
#summarize mean +/- std across folds, grouped by PCA dimensionality (and the deliberately poor
#raw-KMeans-labeled arm), for the paper
summary = results_df.groupby('n_components', sort=False)[['ari', 'svm_auc_pseudo', 'svm_auc_truth', 'rf_auc_pseudo', 'rf_auc_truth']].agg(['mean', 'std'])
print(f"=== {N_FOLDS}-Fold Patient-Grouped CV Summary (grouped by configuration) ===")
print(summary)

summary.to_csv('../data/processed/cv_robustness_summary.csv')
results_df.to_csv('../data/processed/cv_robustness_per_fold.csv', index=False)

=== 5-Fold Patient-Grouped CV Summary (grouped by configuration) ===
                               ari           svm_auc_pseudo            \
                              mean       std           mean       std   
n_components                                                            
2                         0.851897  0.018903       0.990915  0.012124   
3                         0.865423  0.043431       0.992046  0.007983   
112                       0.853074  0.029812       0.984496  0.013936   
raw_kmeans_label_on_pca3  0.437580  0.405913       0.764126  0.392054   

                         svm_auc_truth           rf_auc_pseudo            \
                                  mean       std          mean       std   
n_components                                                               
2                             0.992909  0.008313      0.980508  0.025765   
3                             0.992893  0.008675      0.982803  0.022805   
112                           1.000000 

In [ ]:
#plot boxplots showing the spread of AUC across folds for each configuration 
LABELS = {
    'gmm_2': 'PCA(2)\n x GMM', 'gmm_3': 'PCA(3)\n x GMM', 'gmm_112': 'PCA(112)\n x GMM',
    'kmeans_2': 'PCA(2)\n x KMeans', 'kmeans_3': 'PCA(3)\n x KMeans', 'kmeans_112': 'PCA(112)\n x KMeans',
    'umap_hier_bad': 'UMAP(30)\n x Hierarchical',
}
auc_cols = ['svm_auc_pseudo', 'svm_auc_truth', 'rf_auc_pseudo', 'rf_auc_truth']
auc_labels = ['SVM\n(ps.)', 'SVM\n(gt.)', 'RF\n(ps.)', 'RF\n(gt.)']
group_order = list(dict.fromkeys(results_df['n_components']))  #preserve first-seen order, no duplicates

n_cols = 4
n_rows = int(np.ceil(len(group_order) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.6 * n_cols, 4.2 * n_rows), sharey=True)
axes = np.array(axes).reshape(n_rows, n_cols)
for idx, group in enumerate(group_order):
    ax = axes[idx // n_cols, idx % n_cols]
    subset = results_df[results_df['n_components'] == group]
    ax.boxplot([subset[c] for c in auc_cols], tick_labels=auc_labels)
    ax.set_title(LABELS[group], fontsize=10)
    ax.grid(True, axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)
for idx in range(len(group_order), n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].axis('off')
for r in range(n_rows):
    axes[r, 0].set_ylabel('AUC')
fig.suptitle(f'{N_FOLDS}-Fold Patient-Grouped CV: AUC Distribution by Configuration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/cv_auc_boxplot.png', dpi=150)
plt.show()

In [ ]:
#show the relationship between per-fold clustering quality (ARI) and downstream classifier performance (AUC)
#trained on pseudo-labels
SHORT_LABELS = {
    'gmm_2': 'PCA(2)xGMM', 'gmm_3': 'PCA(3)xGMM', 'gmm_112': 'PCA(112)xGMM',
    'kmeans_2': 'PCA(2)xKMeans', 'kmeans_3': 'PCA(3)xKMeans', 'kmeans_112': 'PCA(112)xKMeans',
    'umap_hier_bad': 'UMAP(30)xHierarchical',
}
group_order = list(dict.fromkeys(results_df['n_components']))
palette = plt.get_cmap('tab10').colors
colors = {group: palette[i % len(palette)] for i, group in enumerate(group_order)}

fig, ax = plt.subplots(figsize=(8, 6))
for group in group_order:
    subset = results_df[results_df['n_components'] == group]
    color = colors[group]
    ax.scatter(subset['ari'], subset['svm_auc_pseudo'], label=f'SVM, {SHORT_LABELS[group]}', s=80, color=color, marker='o')
    ax.scatter(subset['ari'], subset['rf_auc_pseudo'], label=f'RF, {SHORT_LABELS[group]}', s=80, color=color, marker='^')
ax.set_xlabel('Clustering ARI (fold-train, vs ground truth)')
ax.set_ylabel('Classifier AUC (fold-val, vs ground truth)')
ax.set_title('Pseudo-Label Quality vs Downstream Classifier Performance')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/cv_ari_vs_auc_scatter.png', dpi=150)
plt.show()

In [ ]:
#check if higher clustering quality (ARI) predict a SMALLER gap between the
#pseudo-label-trained and ground-truth-trained classifier.
results_df['svm_auc_gap'] = (results_df['svm_auc_truth'] - results_df['svm_auc_pseudo']).abs()
results_df['rf_auc_gap'] = (results_df['rf_auc_truth'] - results_df['rf_auc_pseudo']).abs()

fig, ax = plt.subplots(figsize=(8, 6))
for group in group_order:
    subset = results_df[results_df['n_components'] == group]
    color = colors[group]
    ari_mean = subset['ari'].mean()
    svm_gap_mean = subset['svm_auc_gap'].mean()
    rf_gap_mean = subset['rf_auc_gap'].mean()
    ax.scatter(ari_mean, svm_gap_mean, s=110, color=color, marker='o', edgecolor='black', linewidth=0.5, label=f'SVM, {SHORT_LABELS[group]}')
    ax.scatter(ari_mean, rf_gap_mean, s=110, color=color, marker='^', edgecolor='black', linewidth=0.5, label=f'RF, {SHORT_LABELS[group]}')

#fit a simple linear trend across all folds/models (raw data, not the aggregated means) to visualize
#the overall relationship
all_ari = pd.concat([results_df['ari'], results_df['ari']])
all_gap = pd.concat([results_df['svm_auc_gap'], results_df['rf_auc_gap']])
if all_ari.nunique() > 1:
    trend_coef = np.polyfit(all_ari, all_gap, 1)
    trend_x = np.linspace(all_ari.min(), all_ari.max(), 50)
    ax.plot(trend_x, np.polyval(trend_coef, trend_x), linestyle='--', color='gray', alpha=0.7,
            label=f'trend (slope={trend_coef[0]:.3f}, fit on all 35 fold-level obs.)')

ax.set_xlabel('Clustering ARI (fold-train, vs ground truth)')
ax.set_ylabel('|AUC(ground truth) - AUC(pseudo-label)|')
ax.set_title('Higher Clustering Quality -> Smaller Pseudo-Label/Ground-Truth AUC Gap')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/cv_ari_vs_auc_gap_scatter.png', dpi=150)
plt.show()

print(f"Correlation between ARI and AUC gap (SVM): {results_df['ari'].corr(results_df['svm_auc_gap']):.4f}")
print(f"Correlation between ARI and AUC gap (RF): {results_df['ari'].corr(results_df['rf_auc_gap']):.4f}")